In [1]:
import ast
import string

import pandas as pd

DATASET_PATH = "../../data/out/splits/random/gpqa/train_distilled_deepseek_v4_flash_extend_w_v4_pro.parquet"

df = pd.read_parquet(DATASET_PATH)
print(f"Path: {DATASET_PATH}")
print(f"Shape: {df.shape}")
df.head(2)

Path: ../../data/out/splits/random/gpqa/train_distilled_deepseek_v4_flash_extend_w_v4_pro.parquet
Shape: (437, 8)


,question,options,answer,answer_index,question_id,distill_ans_correct,distill_reasoning,distill_answer
0,Two quantum states with energies E1 and E2 hav...,"['10^-4 eV', '10^-11 eV', '10^-8 eV\n', '10^-9...",A,0,1,True,"We are given: ""Two quantum states with energie...",a
1,how many of the following compounds exhibit op...,"['3', '4', '5', '6']",B,1,3,False,,


## Accuracy


In [2]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

Accuracy: 0.8169 (357/437)


## Missing distilled reasoning traces


In [3]:
missing_nan = int(df["distill_reasoning"].isna().sum())
empty_mask = df["distill_reasoning"].fillna("").str.strip() == ""
missing_or_empty = int(empty_mask.sum())
print(f"NaN distill_reasoning:            {missing_nan}")
print(f"Missing or empty distill_reasoning: {missing_or_empty}")

NaN distill_reasoning:            0
Missing or empty distill_reasoning: 28


## Reasoning trace length distribution (chars)


In [4]:
existing_reasoning = df.loc[~empty_mask, "distill_reasoning"]
lengths = existing_reasoning.str.len()
print(lengths.describe())
print(f"P60: {lengths.quantile(0.6)}")
print(f"P70: {lengths.quantile(0.7)}")
print(f"P80: {lengths.quantile(0.8)}")
print(f"P85: {lengths.quantile(0.85)}")
print(f"P90: {lengths.quantile(0.9)}")
print(f"P92: {lengths.quantile(0.92)}")
print(f"P95: {lengths.quantile(0.95)}")
print(f"P99: {lengths.quantile(0.99)}")

count       409.000000
mean      15304.982885
std       17917.975071
min         201.000000
25%        2688.000000
50%        8287.000000
75%       20588.000000
max      122343.000000
Name: distill_reasoning, dtype: float64
P60: 12276.599999999995
P70: 18636.399999999998
P80: 27076.600000000006
P85: 33884.6
P90: 40636.19999999999
P92: 44808.320000000014
P95: 51921.799999999996
P99: 65977.28


In [ ]:
# corrected_empty_mask = df["corrected_reasoning"].fillna("").str.strip() == ""
# existing_corrected = df.loc[~corrected_empty_mask, "corrected_reasoning"]
# lengths = existing_corrected.str.len()
# print(lengths.describe())
# print(f"P60: {lengths.quantile(0.6)}")
# print(f"P70: {lengths.quantile(0.7)}")
# print(f"P80: {lengths.quantile(0.8)}")
# print(f"P85: {lengths.quantile(0.85)}")
# print(f"P90: {lengths.quantile(0.9)}")
# print(f"P92: {lengths.quantile(0.92)}")
# print(f"P95: {lengths.quantile(0.95)}")
# print(f"P99: {lengths.quantile(0.99)}")

count    1051.000000
mean      185.167460
std       141.933762
min        10.000000
25%        88.000000
50%       149.000000
75%       248.000000
max      1082.000000
Name: corrected_reasoning, dtype: float64
P60: 179.0
P70: 225.0
P80: 271.0
P85: 305.5
P90: 361.0
P92: 393.0
P95: 441.0
P99: 687.0


## Reasoning trace length distribution (tokens)

Token counts per reasoning trace under each of the three model tokenizers we use: `llama3b` (`meta-llama/Llama-3.2-3B-Instruct`), `phi4mini` (`microsoft/Phi-4-mini-instruct`), and `qwen3b` (`Qwen/Qwen2.5-3B-Instruct`).

In [27]:
# from tqdm.auto import tqdm
# from transformers import AutoTokenizer

# MODEL_TOKENIZERS = {
#     "llama3b": "meta-llama/Llama-3.2-3B-Instruct",
#     "phi4mini": "microsoft/Phi-4-mini-instruct",
#     "qwen3b": "Qwen/Qwen2.5-3B-Instruct",
# }

# reasoning_texts = existing_reasoning.tolist()

# for name, model_id in MODEL_TOKENIZERS.items():
#     tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
#     token_lengths = pd.Series(
#         [len(tokenizer(text, add_special_tokens=False)["input_ids"]) for text in tqdm(reasoning_texts, desc=name)],
#         index=existing_reasoning.index,
#     )
#     print(f"=== {name} ({model_id}) ===")
#     print(token_lengths.describe())
#     print(f"P60: {token_lengths.quantile(0.6)}")
#     print(f"P70: {token_lengths.quantile(0.7)}")
#     print(f"P80: {token_lengths.quantile(0.8)}")
#     print(f"P85: {token_lengths.quantile(0.85)}")
#     print(f"P90: {token_lengths.quantile(0.9)}")
#     print(f"P92: {token_lengths.quantile(0.92)}")
#     print(f"P95: {token_lengths.quantile(0.95)}")
#     print(f"P99: {token_lengths.quantile(0.99)}")
#     print()

## Invalid answers

An answer is valid iff, after stripping/lowercasing, it is one of the option letters `a`, `b`, ... up to the row's number of options.


In [ ]:
LETTERS = list(string.ascii_lowercase)


def n_options(opts):
    if isinstance(opts, str):
        try:
            return len(ast.literal_eval(opts))
        except Exception:
            return 0
    try:
        return len(opts)
    except TypeError:
        return 0


def is_valid_answer(ans, n_opts):
    if not isinstance(ans, str):
        return False
    return ans.strip().lower() in set(LETTERS[:n_opts])


# n_opts_series = df["options"].apply(n_options)
valid_mask = df["distill_ans_correct"]
n_invalid = int((~valid_mask).sum())
print(f"Invalid distill_answer count: {n_invalid} ({n_invalid / n:.2%})")

Invalid distill_answer count: 5 (0.07%)


### All invalid answers


In [60]:
invalid_answers = df.loc[~valid_mask, "distill_answer"]
for idx, ans in invalid_answers.items():
    print(f"--- row {idx} ---")
    print(repr(ans))

--- row 3144 ---
'314'
--- row 3176 ---
''
--- row 3274 ---
'\\(2\\sqrt{3}\\) cm'
--- row 3997 ---
'1, 9'
--- row 7122 ---
'15'


### 10 shortest reasoning traces


In [5]:
shortest = existing_reasoning.str.len().nsmallest(30)
for idx, length in shortest.items():
    print(f"--- row {idx} (len={length}) ---")
    print(df.loc[idx, "distill_reasoning"])
    print()

--- row 6309 (len=51) ---
We need answer fraction. (3/4)^5 = 243/1024. Final.

--- row 1608 (len=52) ---
We need answer only final. Need compute 27+9+3+1=40.

--- row 5992 (len=53) ---
We need answer only. Need compute 2. Need final only.

--- row 1862 (len=54) ---
We need answer only. Expected = (1+...+8)/8 =36/8=4.5.

--- row 2455 (len=55) ---
We need answer. Need compute C(5,1)=5. Must final only.

--- row 113 (len=56) ---
We need answer only final. Need compute g(5)=9, f(9)=82.

--- row 6511 (len=56) ---
We need answer solve. 5-x=8 => -x=3 => x=-3. Final only.

--- row 537 (len=57) ---
We need answer. Need compute 3*2+5*7=6+35=41. Final only.

--- row 2322 (len=57) ---
We need answer percent. 60% * 25% = 15%. Need final only.

--- row 589 (len=58) ---
We need answer only final. Need compute (2+5)/(1+10)=7/11.

--- row 6602 (len=58) ---
We need answer simple. sqrt(180)=sqrt(36*5)=6sqrt5. Final.

--- row 974 (len=59) ---
We need answer only. (2^2)^3 = 4^3=64. Need LaTeX maybe 64.

--

## Reset distill columns for invalid answers

Sets `distill_reasoning` and `distill_answer` to `""` and `distill_ans_correct` to `False` for rows where the answer is invalid, then writes back to `DATASET_PATH`.


In [31]:
# invalid_idx = df.index[~valid_mask]
# df.loc[invalid_idx, "distill_reasoning"] = ""
# df.loc[invalid_idx, "distill_answer"] = ""
# df.loc[invalid_idx, "distill_ans_correct"] = False

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(invalid_idx)} rows and wrote {DATASET_PATH}")

## Reset reasoning traces shorter than 10 chars

Sets `distill_reasoning` to `""` for rows where the existing trace is shorter than 10 characters, then writes back to `DATASET_PATH`.

In [32]:
# short_mask = df["distill_reasoning"].fillna("").str.len() < 200
# short_idx = df.index[short_mask]
# df.loc[short_idx, "distill_reasoning"] = ""
# df.loc[short_idx, "distill_answer"] = ""
# df.loc[short_idx, "distill_ans_correct"] = False
# print(f"distill_reasoning rows under {200} chars: {len(short_idx)}")

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")

In [6]:
max_reasoning_chars = 8192
long_mask = df["distill_reasoning"].str.len() > max_reasoning_chars
print(f"distill_reasoning rows over {max_reasoning_chars} chars: {int(long_mask.sum())}")
df.loc[long_mask, "distill_reasoning"] = (
    df.loc[long_mask, "distill_reasoning"].str.slice(stop=max_reasoning_chars)
)
# df.loc[long_mask, "distill_reasoning"] = ""
# df.loc[long_mask, "distill_answer"] = ""
# df.loc[long_mask, "distill_ans_correct"] = False

distill_reasoning rows over 8192 chars: 326


In [7]:
missing_reasoning_mask = df["distill_reasoning"].fillna("").eq("")
print(f"Dropping {int(missing_reasoning_mask.sum())} rows with NA or empty distill_reasoning")
df = df.loc[~missing_reasoning_mask].copy()

Dropping 1 rows with NA or empty distill_reasoning


In [64]:
max_corrected_reasoning_chars = 1444
max_combined_reasoning_chars = 8192

# df["corrected_reasoning"] = df["corrected_reasoning"].str.slice(
#     stop=max_corrected_reasoning_chars
# )
has_corrected_reasoning = df["corrected_reasoning"].fillna("").str.strip().ne("")
df.loc[has_corrected_reasoning, "distill_reasoning"] = [
    (reasoning if isinstance(reasoning, str) else "")[:
        max_combined_reasoning_chars - len(corrected)
    ] + corrected
    for reasoning, corrected in zip(
        df.loc[has_corrected_reasoning, "distill_reasoning"],
        df.loc[has_corrected_reasoning, "corrected_reasoning"],
    )
]

In [8]:
df.to_parquet(
    "../../data/out/distillation/hendrycks_math_distilled_deepseek_v4_flash_regenerate_incorrect_w_large_head8192.parquet", index=False
)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")